# Лекция и методическое руководство: Агрегация данных и объединение таблиц в SQL
## Конспект преподавателя (для объяснения у доски и живой демонстрации)

> **Как пользоваться этим блокнотом на паре:**
> * Блоки **«📌 На доску»** содержат емкие синтаксические конструкции и правила для переноса на доску мелом/маркером.
> * Блоки **«💡 Зачем это нужно»** дают понятные аналогии для объяснения физического смысла студентам.
> * Блоки **«⚙️ Как это работает»** раскрывают поведение СУБД под капотом (обработка NULL, порядок операций).
> * Блоки **«💻 Демонстрация»** содержат готовые минимальные примеры кода для запуска на проекторе в реальном времени.
> * Блоки **«⚠️ Грабли новичков»** перечисляют типовые ошибки из практики и лекторских документов, которые студенты гарантированно сделают, если их не предупредить.

---

### План занятия (тайминг на 90 минут):
1. **Пять агрегатных функций и арифметика `NULL`** (15 мин)
2. **Группировка строк (`GROUP BY`) и правило проекции `SELECT`** (15 мин)
3. **Фильтрация вычисленных групп (`HAVING`) против фильтра строк (`WHERE`)** (15 мин)
4. **Физический порядок выполнения SQL-запроса (SQL Pipeline)** (10 мин)
5. **Внутреннее соединение (`INNER JOIN`) и псевдонимы таблиц** (15 мин)
6. **Левое внешнее соединение (`LEFT JOIN`) и поиск сирот (`WHERE IS NULL`)** (10 мин)
7. **Комбинация `JOIN + GROUP BY + HAVING` для аналитических отчетов** (10 мин)


In [1]:
import sqlite3
import pandas as pd

# Создаем базу в оперативной памяти для интерактивных демонстраций у доски
conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")
cur = conn.cursor()

# Создаем демонстрационную схему мини-клуба
cur.executescript('''
CREATE TABLE demo_members (
    member_id INTEGER PRIMARY KEY,
    nickname TEXT NOT NULL,
    city TEXT NOT NULL,
    loyalty_tier TEXT NOT NULL
);

CREATE TABLE demo_pcs (
    pc_id INTEGER PRIMARY KEY,
    zone_name TEXT NOT NULL,
    hourly_rate INTEGER NOT NULL
);

CREATE TABLE demo_sessions (
    session_id INTEGER PRIMARY KEY,
    member_id INTEGER REFERENCES demo_members(member_id),
    pc_id INTEGER REFERENCES demo_pcs(pc_id),
    game_title TEXT NOT NULL,
    hours_played REAL NOT NULL,
    total_paid INTEGER NOT NULL,
    rating INTEGER
);

INSERT INTO demo_members VALUES
(1, 'Shadow', 'Москва', 'Gold'),
(2, 'Viper', 'Санкт-Петербург', 'Silver'),
(3, 'Neo', 'Москва', 'Platinum'),
(4, 'Ghost', 'Казань', 'Bronze'),
(5, 'Frost', 'Москва', 'Bronze'); -- Клиент без сессий (спящий!)

INSERT INTO demo_pcs VALUES
(101, 'Standard', 150),
(102, 'Standard', 150),
(103, 'VIP', 400),
(104, 'Bootcamp', 250); -- ПК без единой сессии (простаивающий!)

INSERT INTO demo_sessions VALUES
(1, 1, 101, 'Dota 2', 2.0, 300, 5),
(2, 1, 103, 'Cyberpunk 2077', 3.0, 1200, 5),
(3, 2, 101, 'Dota 2', 1.0, 150, 4),
(4, 2, 102, 'Counter-Strike 2', 4.0, 600, NULL), -- Отзыв не оставлен!
(5, 3, 103, 'Counter-Strike 2', 2.5, 1000, 5),
(6, NULL, 102, 'Dota 2', 1.5, 225, 3); -- Анонимный гость без клубной карты!
''')
conn.commit()

print("✅ Демонстрационная база готова к показу.")


✅ Демонстрационная база готова к показу.


---
## Тема 1. Пять агрегатных функций и арифметика `NULL`

### 📌 На доску
```sql
SELECT 
    COUNT(*)                AS total_rows,     -- Считает ВСЕ строки (включая NULL)
    COUNT(rating)           AS rated_rows,     -- Считает только строки, где rating НЕ NULL
    SUM(total_paid)         AS total_revenue,  -- Сумма значений
    AVG(total_paid)         AS avg_check,      -- Среднее арифметическое
    MIN(hours_played)       AS min_hours,      -- Минимальное значение
    MAX(total_paid)         AS max_payment     -- Максимальное значение
FROM demo_sessions;
```

### 💡 Зачем это нужно
* В реальном бизнесе руководителю не нужны 100 000 строк чеков по отдельности. Ему нужна **сводка**: сколько заработали за день, какой средний чек и какая минимальная/максимальная нагрузка.
* Агрегатная функция «схлопывает» множество строк в одно итоговое скалярное значение.

### ⚙️ Как это работает
* **Главное правило `NULL`:** все агрегатные функции (`SUM`, `AVG`, `MIN`, `MAX`, `COUNT(col)`) **полностью игнорируют `NULL`**.
* Если в таблице 10 сессий, но оценку поставили только 8 человек, `AVG(rating)` посчитает среднее по 8 числам (деление на 8, а не на 10).
* Единственное исключение — `COUNT(*)`, который считает физические строки таблицы независимо от наличия пропусков.

### ⚠️ Грабли новичков
* Путаница `COUNT(*)` и `COUNT(column)`. Если столбец содержит `NULL`, `COUNT(column)` вернет меньшее число!
* Попытка использовать агрегат в `WHERE` (об этом подробнее в Теме 3).


In [2]:
# Демонстрация 1: Работа агрегатных функций и игнорирование NULL
query_agg = '''
SELECT 
    COUNT(*) AS "Всего сессий (COUNT *)",
    COUNT(rating) AS "С оценками (COUNT rating)",
    COUNT(*) - COUNT(rating) AS "Пропущено отзывов",
    SUM(total_paid) AS "Выручка (SUM)",
    ROUND(AVG(total_paid), 2) AS "Средний чек (AVG)",
    MIN(total_paid) AS "Минимум (MIN)",
    MAX(total_paid) AS "Максимум (MAX)"
FROM demo_sessions;
'''
pd.read_sql_query(query_agg, conn)


,Всего сессий (COUNT *),С оценками (COUNT rating),Пропущено отзывов,Выручка (SUM),Средний чек (AVG),Минимум (MIN),Максимум (MAX)
0,6,5,1,3475,579.17,150,1200


---
## Тема 2. Группировка данных (`GROUP BY`) и правило проекции `SELECT`

### 📌 На доску
```sql
SELECT game_title, COUNT(*) AS sessions_count, SUM(total_paid) AS revenue
FROM demo_sessions
GROUP BY game_title;
```
> **Золотое правило `GROUP BY`:**  
> Каждый столбец в `SELECT` обязан либо:  
> 1) Присутствовать в выражении `GROUP BY`,  
> 2) Либо находиться внутри агрегатной функции (`COUNT`, `SUM`, `AVG` и т.д.).

### 💡 Зачем это нужно
* Агрегаты по всей таблице дают одну общую цифру на весь клуб. Но менеджеру нужно сравнить **сегменты**: какая игра популярнее? какой тариф выгоднее? из какого города больше выручка?
* `GROUP BY` разбивает таблицу на непересекающиеся «корзины» по уникальным значениям столбца, и вычисляет агрегаты отдельно внутри каждой корзины.

### ⚙️ Как это работает
1. СУБД сканирует строки.
2. Строки с одинаковым значением поля группировки объединяются в одну группу.
3. В каждой группе параллельно вычисляются указанные агрегаты.
4. На выход выдается ровно по одной строке на каждую группу.

### ⚠️ Грабли новичков
* Забыли указать поле в `GROUP BY`: если написать `SELECT game_title, pc_id, COUNT(*) ... GROUP BY game_title`, то в SQLite значение `pc_id` будет взято случайно из любой строки группы, что приводит к непредсказуемым результатам (в других СУБД, например PostgreSQL, это вызовет фатальную ошибку синтаксиса).


In [10]:
# Демонстрация 2: Группировка по играм с вычислением показателей
query_group = '''
SELECT 
    game_title,
    COUNT(*) AS sessions_count,
    SUM(hours_played) AS total_hours,
    SUM(total_paid) AS total_revenue,
    ROUND(AVG(rating), 2) AS avg_rating
FROM demo_sessions
GROUP BY game_title
ORDER BY total_revenue DESC;
'''
pd.read_sql_query(query_group, conn)


,game_title,sessions_count,total_hours,total_revenue,avg_rating
0,Counter-Strike 2,2,6.5,1600,5.0
1,Cyberpunk 2077,1,3.0,1200,5.0
2,Dota 2,3,4.5,675,4.0


---
## Тема 3. Фильтрация вычисленных групп (`HAVING`) против фильтра строк (`WHERE`)

### 📌 На доску
| Критерий | `WHERE` | `HAVING` |
|---|---|---|
| **Момент фильтрации** | **ДО** группировки (фильтрует исходные строки) | **ПОСЛЕ** группировки (фильтрует готовые группы) |
| **С какими полями работает** | С физическими столбцами таблицы | С вычисленными агрегатами (`COUNT`, `AVG`, `SUM`) |
| **Допустимость агрегатов** | ❌ **ЗАПРЕЩЕНО** (`WHERE COUNT(*) > 5` — ошибка!) | ✔️ **РАЗРЕШЕНО** (`HAVING COUNT(*) > 5`) |

```sql
-- ПРАВИЛЬНО:
SELECT game_title, COUNT(*) AS cnt, SUM(total_paid) AS rev
FROM demo_sessions
WHERE hours_played >= 2.0        -- 1. Сначала отсекаем короткие сессии
GROUP BY game_title              -- 2. Группируем оставшиеся
HAVING SUM(total_paid) > 500;    -- 3. Оставляем только прибыльные группы

-- ОШИБКА:
-- SELECT game_title FROM demo_sessions WHERE COUNT(*) > 2;  <-- НЕЛЬЗЯ!
```

### 💡 Зачем это нужно
* Когда мы пишем `WHERE`, группы еще не сформированы — СУБД физически не знает, сколько строк попадет в группу.
* Поэтому для отсечения групп по их вычисленным свойствам (например, «оставить курсы/игры, где выручка > 5000») придумали отдельное ключевое слово `HAVING`.

### ⚙️ Как это работает
* Сначала `WHERE` просеивает строки таблицы.
* Оставшиеся строки собираются в группы по `GROUP BY`.
* Для каждой группы рассчитываются агрегатные функции.
* `HAVING` отбрасывает те группы, для которых агрегатное условие ложно.

### ⚠️ Грабли новичков
* Использование `HAVING` вместо `WHERE` для простых столбцов (например, `HAVING hours_played > 2`). Это неэффективно, так как заставляет СУБД группировать лишние строки.
* Попытка написать агрегатную функцию внутри `WHERE`.


In [4]:
# Демонстрация 3: Совместное использование WHERE + GROUP BY + HAVING
# Задача: найти игры, где сессии длились от 2 часов и более, 
# и суммарная выручка по этим длинным сессиям превысила 500 рублей
query_having = '''
SELECT 
    game_title,
    COUNT(*) AS long_sessions,
    SUM(total_paid) AS group_revenue
FROM demo_sessions
WHERE hours_played >= 2.0
GROUP BY game_title
HAVING SUM(total_paid) > 500;
'''
pd.read_sql_query(query_having, conn)


,game_title,long_sessions,group_revenue
0,Counter-Strike 2,2,1600
1,Cyberpunk 2077,1,1200


---
## Тема 4. Физический порядок выполнения SQL-запроса (SQL Pipeline)

### 📌 На доску
```
   1. FROM         (Откуда берем таблицы и как их соединяем JOIN)
      ↓
   2. WHERE        (Строковая фильтрация ДО группировки)
      ↓
   3. GROUP BY     (Формирование групп)
      ↓
   4. HAVING       (Фильтрация групп по агрегатным условиям)
      ↓
   5. SELECT       (Вычисление выражений, применение псевдонимов AS)
      ↓
   6. ORDER BY     (Сортировка итогового набора данных)
      ↓
   7. LIMIT/OFFSET (Пагинация и ограничение вывода)
```

### 💡 Почему это важно понимать
* В коде мы пишем `SELECT` первым словом, но СУБД выполняет его **пятым** по счету!
* Именно поэтому:
  1. Нельзя использовать псевдоним из `SELECT` внутри блока `WHERE` (на этапе `WHERE` псевдоним еще не вычислен!).
  2. Нельзя использовать агрегатные функции в `WHERE` (на этапе `WHERE` группы еще не существуют!).
  3. В `ORDER BY` и `HAVING` псевдонимы из `SELECT` использовать уже можно, так как они идут позже.


---
## Тема 5. Внутреннее соединение (`INNER JOIN`) и псевдонимы таблиц

### 📌 На доску
```sql
SELECT 
    s.session_id,
    m.nickname,
    m.city,
    s.game_title,
    s.total_paid
FROM demo_sessions s
JOIN demo_members m ON s.member_id = m.member_id;
```
> **Суть INNER JOIN:**  
> В результат попадают **только те строки, для которых нашлось строгое совпадение по ключу в обеих таблицах**.  
> Если `member_id` равен `NULL` (аноним) — строка в результат **не попадет**.  
> Если клиент зарегистрирован, но ни разу не играл — он в результат **не попадет**.

### 💡 Зачем это нужно
* В нормализованной базе в таблице `demo_sessions` лежит число `member_id = 1`. Имени клиента там нет, чтобы не дублировать память.
* Чтобы собрать читаемый отчет для человека («Клиент Shadow из Москвы сыграл в Dota 2 на 300 руб.»), мы сшиваем строки двух таблиц по первичному и внешнему ключам.

### ⚙️ Как это работает
* СУБД сопоставляет строки левой таблицы со строками правой таблицы по предикату `ON`.
* Использование коротких псевдонимов (`s` для сессий, `m` для клиентов) делает код компактным и защищает от конфликтов, если в обеих таблицах есть столбцы с одинаковым именем (например, `id` или `status`).

### ⚠️ Грабли новичков
* Забыли блок `ON`: получается **Декартово произведение** (каждая строка с каждой, количество строк перемножается, сервер виснет).
* Соединение по неверному столбцу (например, перепутали `pc_id` и `member_id`).


In [5]:
# Демонстрация 5: Внутреннее соединение сессий с зарегистрированными клиентами
query_inner = '''
SELECT 
    s.session_id,
    m.nickname AS player_name,
    m.city,
    m.loyalty_tier,
    s.game_title,
    s.total_paid
FROM demo_sessions s
JOIN demo_members m ON s.member_id = m.member_id
ORDER BY s.total_paid DESC;
'''
pd.read_sql_query(query_inner, conn)


,session_id,player_name,city,loyalty_tier,game_title,total_paid
0,2,Shadow,Москва,Gold,Cyberpunk 2077,1200
1,5,Neo,Москва,Platinum,Counter-Strike 2,1000
2,4,Viper,Санкт-Петербург,Silver,Counter-Strike 2,600
3,1,Shadow,Москва,Gold,Dota 2,300
4,3,Viper,Санкт-Петербург,Silver,Dota 2,150


---
## Тема 6. Левое соединение (`LEFT JOIN`) и паттерн поиска сирот (`WHERE IS NULL`)

### 📌 На доску
```sql
-- 1. Все клиенты и их сессии (даже если сессий 0):
SELECT m.nickname, m.city, s.session_id, s.total_paid
FROM demo_members m
LEFT JOIN demo_sessions s ON m.member_id = s.member_id;

-- 2. Классический паттерн поиска "сирот" / "спящих" клиентов:
SELECT m.nickname, m.city, m.loyalty_tier
FROM demo_members m
LEFT JOIN demo_sessions s ON m.member_id = s.member_id
WHERE s.session_id IS NULL;  -- <--- Ключевой фильтр!
```

### 💡 Зачем это нужно
* `INNER JOIN` безжалостно отбрасывает всё, у чего нет пары. Но маркетологу клуба критически важно найти **клиентов, которые зарегистрировались, но ни разу не пришли играть** (чтобы отправить им промокод!).
* А техническому директору нужно найти **компьютеры, на которых за месяц не запустили ни одной игры** (чтобы проверить, не сломаны ли они).
* Для этого используют `LEFT JOIN`: левая таблица сохраняется **полностью**, а если справа совпадений нет, все поля правой таблицы заполняются значением `NULL`.

### ⚙️ Как это работает
1. Берется левая таблица (`demo_members`).
2. Для каждой строки ищется соответствие в правой таблице.
3. Если соответствие есть — строка дополняется данными справа.
4. Если соответствия нет — строка остается, а поля правой таблицы становятся `NULL`.
5. Условие `WHERE right_table.id IS NULL` оставляет только те строки, у которых пары не нашлось.

### ⚠️ Главная ловушка `LEFT JOIN`
> **Опасность `WHERE`:**  
> Если написать `LEFT JOIN demo_sessions s ON ... WHERE s.game_title = 'Dota 2'`, то для клиентов без сессий поле `s.game_title` равно `NULL`.  
> Сравнение `NULL = 'Dota 2'` вернет `FALSE`, и эти клиенты отсекутся!  
> В итоге `LEFT JOIN` превратится в обычный `INNER JOIN`.  
> **Правило:** фильтры правой таблицы в `LEFT JOIN` нужно писать **внутри `ON`**, а не в `WHERE`!


In [14]:
# Демонстрация 6: Поиск спящих клиентов (LEFT JOIN + WHERE IS NULL)
query_sleeping = '''
SELECT 
    m.member_id,
    m.nickname,
    m.city,
    m.loyalty_tier,
    s.session_id AS "ID сессии (NULL = нет сессий)"
FROM demo_members m
LEFT JOIN demo_sessions s ON m.member_id = s.member_id
WHERE s.session_id IS NULL;
'''
pd.read_sql_query(query_sleeping, conn)


,member_id,nickname,city,loyalty_tier,ID сессии (NULL = нет сессий)
0,4,Ghost,Казань,Bronze,None
1,5,Frost,Москва,Bronze,None


In [7]:
# Демонстрация 6.2: Поиск компьютеров без сессий (простаивающее оборудование)
query_idle_pcs = '''
SELECT 
    p.pc_id,
    p.zone_name,
    p.hourly_rate,
    s.session_id
FROM demo_pcs p
LEFT JOIN demo_sessions s ON p.pc_id = s.pc_id
WHERE s.session_id IS NULL;
'''
pd.read_sql_query(query_idle_pcs, conn)


,pc_id,zone_name,hourly_rate,session_id
0,104,Bootcamp,250,None


---
## Тема 7. Комбинирование `JOIN + GROUP BY + HAVING` для сводной аналитики

### 📌 На доску
```sql
SELECT 
    m.nickname,
    m.loyalty_tier,
    COUNT(s.session_id) AS total_sessions,
    SUM(s.total_paid)   AS customer_ltv
FROM demo_members m
JOIN demo_sessions s ON m.member_id = s.member_id
GROUP BY m.member_id, m.nickname, m.loyalty_tier
HAVING SUM(s.total_paid) >= 500
ORDER BY customer_ltv DESC;
```

### 💡 Зачем это нужно
* В реальных аналитических отчетах мы почти никогда не группируем одну «голую» таблицу.
* Настоящие бизнес-отчеты строятся на стыке таблиц: берем имя клиента из справочника `members`, соединяем с журналом `game_sessions`, группируем по клиенту и считаем его суммарный LTV (выручку за все время).

### ⚙️ Полная цепочка из 3 таблиц
* `demo_sessions` соединяется с `demo_members` (чтобы знать, кто играл) и с `demo_pcs` (чтобы знать, в какой зоне клуба стоял ПК):
```sql
SELECT 
    p.zone_name,
    COUNT(s.session_id) AS sessions_in_zone,
    SUM(s.total_paid)   AS zone_revenue
FROM demo_sessions s
JOIN demo_pcs p ON s.pc_id = p.pc_id
GROUP BY p.zone_name;
```


In [8]:
# Демонстрация 7: Сквозная доходность игровых зон (JOIN + GROUP BY)
query_zone_kpi = '''
SELECT 
    p.zone_name,
    COUNT(s.session_id) AS total_sessions,
    SUM(s.hours_played) AS hours_billed,
    SUM(s.total_paid) AS total_revenue,
    ROUND(AVG(s.total_paid), 2) AS avg_ticket
FROM demo_pcs p
JOIN demo_sessions s ON p.pc_id = s.pc_id
GROUP BY p.zone_name
ORDER BY total_revenue DESC;
'''
pd.read_sql_query(query_zone_kpi, conn)


,zone_name,total_sessions,hours_billed,total_revenue,avg_ticket
0,VIP,2,5.5,2200,1100.00
1,Standard,4,8.5,1275,318.75


---
## Чек-лист для преподавателя перед раздачей заданий студентам:
1. [ ] Проверить, что студенты помнят: агрегаты в `WHERE` писать **нельзя**.
2. [ ] Убедиться, что студенты понимают разницу между `COUNT(*)` и `COUNT(column)`.
3. [ ] Напомнить, что поиск «потерянных» записей выполняется строго через `LEFT JOIN ... WHERE right.key IS NULL`.
4. [ ] Предупредить, что все пороговые числа в запросах должны передаваться через плейсхолдеры `?`.
5. [ ] Напомнить закрыть базу в конце работы: `conn.close()`.
